# PCT Evaluation of lean models

Load and test trained models, and show various evaluation metrics.

Note: Does no longer use original PCT data.

## Prep

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DATASET_PATH = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/datasets'
RESULTS_PATH = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/results'
CHECKPOINTS_PATH = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/checkpoints'

os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
# Get repo
!git clone --recursive --branch lean-model https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}
%cd {REPO_PATH}
%pip install -r envs/pct/requirements.txt

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 743, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 743 (delta 51), reused 75 (delta 32), pack-reused 619 (from 2)
Receiving objects: 100% (743/743), 69.95 MiB | 42.41 MiB/s, done.
Resolving deltas: 100% (328/328), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 178, done.        
remote: Counting objects: 100% (27/27), done.        
remote: Compressing objects: 100% (21/21), done.        
remote: Total 178 (delta 16), reused 16 (delta 6), pack-reused 151 (from 1)        
Receiving objects: 100% (178/178), 9.68 MiB | 32.71 MiB/s, done.
Resolving deltas: 100% (8

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from IPython.display import display, HTML
from tqdm import tqdm

In [5]:
# Copy select files from Google Drive to the repo dataset folder
!mkdir /content/downloads
!rsync -avP {DATASET_PATH}/fullmodelnet40.tar.gz /content/downloads
!rsync -avP {DATASET_PATH}/partialmodelnet40.tar.gz /content/downloads
!rsync -avP {DATASET_PATH}/pm40_extreme_partiality.tar.gz /content/downloads

# Extract zip files
!tar -xvzf /content/downloads/fullmodelnet40.tar.gz -C {REPO_PATH}/datasets
!tar -xvzf /content/downloads/partialmodelnet40.tar.gz -C {REPO_PATH}/datasets
!tar -xvzf /content/downloads/pm40_extreme_partiality.tar.gz -C {REPO_PATH}/datasets

sending incremental file list
fullmodelnet40.tar.gz
  1,833,210,387 100%   76.85MB/s    0:00:22 (xfr#1, to-chk=0/1)

sent 1,833,658,058 bytes  received 35 bytes  74,843,187.47 bytes/sec
total size is 1,833,210,387  speedup is 1.00
sending incremental file list
partialmodelnet40.tar.gz
  2,162,130,396 100%   82.93MB/s    0:00:24 (xfr#1, to-chk=0/1)

sent 2,162,658,366 bytes  received 35 bytes  84,810,133.37 bytes/sec
total size is 2,162,130,396  speedup is 1.00
sending incremental file list
pm40_extreme_partiality.tar.gz
  8,334,013,260 100%   79.93MB/s    0:01:39 (xfr#1, to-chk=0/1)

sent 8,336,048,044 bytes  received 35 bytes  82,128,552.50 bytes/sec
total size is 8,334,013,260  speedup is 1.00
fullmodelnet40/
fullmodelnet40/test_filenames.txt
fullmodelnet40/test_gt_rot.npy
fullmodelnet40/test_gt_tra.npy
fullmodelnet40/test_labels.npy
fullmodelnet40/test_points.npy
fullmodelnet40/train_filenames.txt
fullmodelnet40/train_gt_rot.npy
fullmodelnet40/train_gt_tra.npy
fullmodelnet40/train_l

## Get Predictions

Upload checkpoints, get test results, visualize and analyze.

You can get the trained model checkpoints from [this folder](https://drive.google.com/drive/folders/1Wo8HHyUl0lcGMnldIeJqdC2i4IsdTFcX?usp=drive_link).

Copy those files into your own Google Drive and, if necessary, change the first path in the code below. Contents:

- Evaluated in this notebook:
- pct-minimal_p50           : Lean PCT model with minimal settings.
- pct-set_decoder_lbrd1_p50 : Lean PCT model with decoder using 1 layer of linear, bn, relu, dropout layer
- pct-set_sa_ch_64_p50      : Lean PCT model with stacked attention using 64 channels
- pct-set_sa_ch_128_p50     : Lean PCT model with stacked attention using 128 channels
- pct-set_sa_layer1_p50     : Lean PCT model with 1 stacked attention layer
- pct-set_sa_stacks1_p50    : Lean PCT model with 1 stacked attention layer stack
- pct-set_sa_stacks2_p50    : Lean PCT model with 2 stacked attention layer stacks
- pct-set_sa_stacks3_p50    : Lean PCT model with 3 stacked attention layer stacks


In [6]:
!rsync -avP {CHECKPOINTS_PATH}/pct*.pth {REPO_PATH}/checkpoints

sending incremental file list
pct-lean.pth
     33,976,300 100%   22.11MB/s    0:00:01 (xfr#1, to-chk=23/24)
pct-mh_p50.pth
     34,642,046 100%   12.01MB/s    0:00:02 (xfr#2, to-chk=22/24)
pct-minimal_p50.pth
      4,783,789 100%    3.38MB/s    0:00:01 (xfr#3, to-chk=21/24)
pct-o_p50.pth
     34,642,686 100%   18.33MB/s    0:00:01 (xfr#4, to-chk=20/24)
pct-o_pct.pth
     34,642,686 100%   11.01MB/s    0:00:03 (xfr#5, to-chk=19/24)
pct-o_pfull.pth
     34,642,686 100%   10.62MB/s    0:00:03 (xfr#6, to-chk=18/24)
pct-p_p50.pth
     34,665,790 100%   12.65MB/s    0:00:02 (xfr#7, to-chk=17/24)
pct-p_pfull.pth
     34,665,790 100%   11.98MB/s    0:00:02 (xfr#8, to-chk=16/24)
pct-po_p50.pth
     34,665,790 100%   12.29MB/s    0:00:02 (xfr#9, to-chk=15/24)
pct-po_pfull.pth
     34,665,790 100%   13.50MB/s    0:00:02 (xfr#10, to-chk=14/24)
pct-set_decoder_lbrd1_p50.pth
     19,779,033 100%   14.55MB/s    0:00:01 (xfr#11, to-chk=13/24)
pct-set_sa_ch_128_p50.pth
      8,725,657 100%    8.63MB/s

In [7]:
%cd {REPO_PATH}/repos/Point-Transformers/
!python test_cls.py --help

/content/pointcloud-bench/repos/Point-Transformers
test_cls is powered by Hydra.

== Configuration groups ==
Compose your configuration from those groups (group=option)

model: Hengshuang, Lean, Menghao, Nico, Patch, lean_minimal, lean_set_decoder_lbrd1, lean_set_sa_ch_128, lean_set_sa_ch_64, lean_set_sa_layer1, lean_set_sa_stacks1, lean_set_sa_stacks2, lean_set_sa_stacks3


== Config ==
Override anything in the config (foo.bar=value)

model:
  name: Menghao
batch_size: 16
epoch: 200
learning_rate: 0.001
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: false
workers: 2
step_size: 50
data_path: ../../datasets/modelnet40_normal_resampled/
checkpoint_path: best_model.pth
partiality: ''
occlusion: false
occlusion_min: 0.1
occlusion_max: 0.9


Powered by Hydra (https://hydra.cc)
Use --hydra-help to view Hydra specific help




## Configs

In [8]:
# Prepare configs
models = ['Patch',
          'lean_minimal',
          'lean_set_decoder_lbrd1',
          'lean_set_sa_ch_128',
          'lean_set_sa_ch_64',
          'lean_set_sa_layer1',
          'lean_set_sa_stacks1',
          'lean_set_sa_stacks2',
          'lean_set_sa_stacks3']
checkpoints = ['pct-p_p50',
               'pct-minimal_p50',
               'pct-set_decoder_lbrd1_p50',
               'pct-set_sa_ch_128_p50',
               'pct-set_sa_ch_64_p50',
               'pct-set_sa_layer1_p50',
               'pct-set_sa_stacks1_p50',
               'pct-set_sa_stacks2_p50',
               'pct-set_sa_stacks3_p50']
test_sets = ["fullmodelnet40", "partialmodelnet40",] + ["pm40_partiality_level"] * 4
partialities = ["", "", "_10", "_20", "_30", "_40"]

# Inference
Load this code once if you still don't have the pickled results file

In [9]:
# Function for inference

from test_cls import get_predictions

config_dir = f'{REPO_PATH}/repos/Point-Transformers/config'

def test_pct(
    data_folder: str = 'partialmodelnet40',
    checkpoint: str = 'pct_p50',
    config_dir: str = config_dir,
    partiality: str = '',
    model: str = ''
) -> tuple[np.array, np.array]:
    """Gets predictions from pretrained PCT model.

    Args:
        data_folder (str, optional): Dataset folder to test on.
            Defaults to 'partialmodelnet40'.
        checkpoint (str, optional): Checkpoint name; omit extension.
            Defaults to 'pct_p50'.
        config_dir (str, optional): Where the model configs reside.
            Only necessary to run, but not used. Defaults to config_dir.
        partiality (str, optional): Used only for extreme partiality
            datasets with non-standard filenames. Use '_10', '_20',
            '_30', '_40'.

    Returns:
        tuple(np.array, np.array): Tuple of predictions and base truths
    """
    if data_folder in ['partialmodelnet40', 'pm40_partiality_level', 'fullmodelnet40']:
        papnet_loader = 'true'
    else: papnet_loader = 'false'

    with initialize_config_dir(config_dir=config_dir, version_base='1.2'):
        cfg = compose(
            config_name='cls',
            overrides=[
                f'data_path=../../datasets/{data_folder}/',
                f'checkpoint_path=../../checkpoints/{checkpoint}.pth',
                f'model={model}',
                f'use_papnet_loader={papnet_loader}',
                f'partiality={partiality}'
            ],
        )
    OmegaConf.set_struct(cfg, False)
    y_true, y_pred = get_predictions(cfg)
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Accuracy: {accuracy:.4f} for dataset {data_folder} and model {checkpoint}')
    return (y_true, y_pred)

In [16]:
import pickle
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm import tqdm

results = {}

for i in tqdm(range(len(models)), desc=f"Running across models..."):
  if models[i] not in results:
    results[models[i]] = {}
  for j in tqdm(range(len(test_sets)), leave=False, desc=f"Running across test sets..."):
    y_true, y_pred = test_pct(
      data_folder=test_sets[j],
      checkpoint=checkpoints[i],
      model=models[i],
      partiality=partialities[j]
    )
    # Compute instance accuracy, per-class accuracy, and mean-class accuracy
    cm = confusion_matrix(y_true, y_pred)
    per_class_acc = cm.diagonal().astype(float) / cm.sum(axis=1)
    if test_sets[j] not in results[models[i]]:
      results[models[i]][test_sets[j]+partialities[j]] = {}
    results[models[i]][test_sets[j]+partialities[j]]["inst_acc"] = accuracy_score(y_true, y_pred)
    results[models[i]][test_sets[j]+partialities[j]]["per_class_acc"] = per_class_acc
    results[models[i]][test_sets[j]+partialities[j]]["mean_class_acc"] = per_class_acc.mean()
    results[models[i]][test_sets[j]+partialities[j]]["confusion_matrix"] = cm

# Write the results to a pickle file

predictions_pkl_path = f'{RESULTS_PATH}/predictions_pct-lean'
if not os.path.exists(predictions_pkl_path):
  os.mkdir(predictions_pkl_path)
with open(os.path.join(predictions_pkl_path, "results.pkl"), 'wb') as file:
    pickle.dump(results, file)

Running across test sets...:   0%|          | 0/6 [00:00<?, ?it/s]

The size of test data is 2468
Total number of parameters: 2876652
Model loaded with 50 epochs



Running across test sets...:  17%|█▋        | 1/6 [00:18<01:31, 18.38s/it]

# iterations (batches): 155
Average inference time per batch: 112.561 ms
Average inference time per sample: 7.069275 ms
Accuracy: 0.1402 for dataset fullmodelnet40 and model pct-p_p50
The size of test data is 2468
Total number of parameters: 2876652
Model loaded with 50 epochs



Running across test sets...:  33%|███▎      | 2/6 [00:34<01:08, 17.23s/it]

...

# iterations (batches): 155
Average inference time per batch: 94.899 ms
Average inference time per sample: 5.960056 ms
Accuracy: 0.2545 for dataset pm40_partiality_level and model pct-set_sa_stacks3_p50
The size of test data is 2468
Total number of parameters: 2055080
Model loaded with 27 epochs



Running across test sets...:  83%|████████▎ | 5/6 [01:17<00:15, 15.39s/it]

# iterations (batches): 155
Average inference time per batch: 95.756 ms
Average inference time per sample: 6.013850 ms
Accuracy: 0.4271 for dataset pm40_partiality_level and model pct-set_sa_stacks3_p50
The size of test data is 2468
Total number of parameters: 2055080
Model loaded with 27 epochs



Running across models...: 100%|██████████| 9/9 [14:03<00:00, 93.68s/it]

# iterations (batches): 155
Average inference time per batch: 94.971 ms
Average inference time per sample: 5.964555 ms
Accuracy: 0.6236 for dataset pm40_partiality_level and model pct-set_sa_stacks3_p50


# Analysis

In [17]:
# Load the pickle file
import numpy as np
import pandas as pd
import pickle

predictions_pkl_path = f'{RESULTS_PATH}/predictions_pct-lean'
with open(os.path.join(predictions_pkl_path, "results.pkl"), 'rb') as f:
    results = pickle.load(f)

## Instance Accuracy

In [18]:
inst_acc_table = pd.DataFrame.from_dict({
  model: {
    test_set: f"{results[model][test_set]['inst_acc']*100:.2f}"
    for test_set in results[model]
  }
  for model in results
})
inst_acc_table = inst_acc_table.transpose()

In [19]:
inst_acc_table

,fullmodelnet40,partialmodelnet40,pm40_partiality_level_10,pm40_partiality_level_20,pm40_partiality_level_30,pm40_partiality_level_40
Patch,14.02,83.27,11.55,27.35,46.47,66.82
lean_minimal,8.39,72.73,9.68,23.46,40.40,57.29
lean_set_decoder_lbrd1,13.49,77.07,10.49,24.80,42.42,61.26
lean_set_sa_ch_128,11.75,74.92,9.72,23.50,40.92,57.90
lean_set_sa_ch_64,5.67,69.85,9.16,21.43,36.22,54.42
lean_set_sa_layer1,11.10,77.03,11.22,25.24,42.79,62.80
lean_set_sa_stacks1,10.17,75.89,11.39,26.34,42.95,60.70
lean_set_sa_stacks2,8.87,76.82,11.35,24.96,42.14,61.67
lean_set_sa_stacks3,10.37,77.59,11.99,25.45,42.71,62.36


## Mean Class Accuracy

In [20]:
mean_class_acc_table = pd.DataFrame.from_dict({
  model: {
    test_set: f"{results[model][test_set]['mean_class_acc']*100:.2f}"
    for test_set in results[model]
  }
  for model in results
})
mean_class_acc_table = mean_class_acc_table.transpose()

In [21]:
mean_class_acc_table

,fullmodelnet40,partialmodelnet40,pm40_partiality_level_10,pm40_partiality_level_20,pm40_partiality_level_30,pm40_partiality_level_40
Patch,14.27,79.26,12.33,25.77,43.78,63.36
lean_minimal,7.78,66.42,10.42,21.46,37.20,52.15
lean_set_decoder_lbrd1,12.63,71.34,11.31,22.79,38.02,56.12
lean_set_sa_ch_128,9.85,69.84,10.44,21.66,38.21,53.20
lean_set_sa_ch_64,6.00,63.73,9.68,20.44,32.53,49.36
lean_set_sa_layer1,10.85,70.93,11.94,23.57,39.82,58.23
lean_set_sa_stacks1,9.18,71.09,11.55,24.40,39.15,56.09
lean_set_sa_stacks2,8.50,71.23,11.12,22.84,38.32,56.34
lean_set_sa_stacks3,9.10,71.82,11.62,23.04,38.62,57.86


# Per Class Accuracies in `testmodelnet40`

In [22]:
per_class_acc_table = pd.DataFrame.from_dict({
  model: results[model]["partialmodelnet40"]["per_class_acc"]
  for model in results
})
per_class_acc_table = per_class_acc_table.transpose()

In [23]:
per_class_acc_table

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
Patch,1.0,0.72,0.92,0.70,0.91,0.96,0.80,0.98,0.98,1.00,...,0.88,0.55,0.70,0.72,0.80,0.96,0.69,0.78,0.30,0.75
lean_minimal,1.0,0.58,0.80,0.50,0.86,0.93,0.75,0.95,0.98,0.85,...,0.78,0.35,0.35,0.73,0.50,0.92,0.50,0.75,0.10,0.55
lean_set_decoder_lbrd1,1.0,0.58,0.85,0.60,0.87,0.92,0.95,0.97,0.98,1.00,...,0.82,0.45,0.65,0.74,0.60,0.96,0.53,0.76,0.00,0.55
lean_set_sa_ch_128,1.0,0.56,0.80,0.65,0.87,0.90,0.90,0.96,0.97,0.95,...,0.79,0.45,0.60,0.74,0.60,0.95,0.45,0.76,0.05,0.50
lean_set_sa_ch_64,1.0,0.36,0.74,0.65,0.85,0.91,0.80,0.97,0.96,0.90,...,0.76,0.20,0.50,0.69,0.45,0.93,0.43,0.72,0.00,0.45
lean_set_sa_layer1,1.0,0.52,0.81,0.60,0.92,0.93,0.90,0.98,0.96,0.90,...,0.81,0.35,0.60,0.80,0.60,0.95,0.60,0.78,0.00,0.45
lean_set_sa_stacks1,1.0,0.58,0.82,0.60,0.87,0.95,0.90,0.95,0.96,0.95,...,0.82,0.40,0.60,0.74,0.70,0.96,0.52,0.78,0.05,0.55
lean_set_sa_stacks2,1.0,0.62,0.85,0.70,0.89,0.95,0.85,0.96,1.00,1.00,...,0.79,0.40,0.55,0.74,0.50,0.95,0.57,0.74,0.00,0.55
lean_set_sa_stacks3,1.0,0.54,0.82,0.60,0.91,0.94,1.00,0.98,0.97,1.00,...,0.82,0.45,0.55,0.80,0.65,0.96,0.56,0.75,0.00,0.65


In [24]:
inst_acc_table.to_latex()

'\\begin{tabular}{lllllll}\n\\toprule\n & fullmodelnet40 & partialmodelnet40 & pm40_partiality_level_10 & pm40_partiality_level_20 & pm40_partiality_level_30 & pm40_partiality_level_40 \\\\\n\\midrule\nPatch & 14.02 & 83.27 & 11.55 & 27.35 & 46.47 & 66.82 \\\\\nlean_minimal & 8.39 & 72.73 & 9.68 & 23.46 & 40.40 & 57.29 \\\\\nlean_set_decoder_lbrd1 & 13.49 & 77.07 & 10.49 & 24.80 & 42.42 & 61.26 \\\\\nlean_set_sa_ch_128 & 11.75 & 74.92 & 9.72 & 23.50 & 40.92 & 57.90 \\\\\nlean_set_sa_ch_64 & 5.67 & 69.85 & 9.16 & 21.43 & 36.22 & 54.42 \\\\\nlean_set_sa_layer1 & 11.10 & 77.03 & 11.22 & 25.24 & 42.79 & 62.80 \\\\\nlean_set_sa_stacks1 & 10.17 & 75.89 & 11.39 & 26.34 & 42.95 & 60.70 \\\\\nlean_set_sa_stacks2 & 8.87 & 76.82 & 11.35 & 24.96 & 42.14 & 61.67 \\\\\nlean_set_sa_stacks3 & 10.37 & 77.59 & 11.99 & 25.45 & 42.71 & 62.36 \\\\\n\\bottomrule\n\\end{tabular}\n'

## Quick Visualization

In [25]:
# Load class names from dataset
class_names_path = os.path.join(REPO_PATH, 'datasets', 'partialmodelnet40', 'partialmodelnet40_shape_names.txt')
with open(class_names_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]
    class_names = {i: name for i, name in enumerate(class_names)}

In [26]:
cm_savepath = os.path.join(RESULTS_PATH, "cm_pct_lean")
if not os.path.exists:
  os.mkdir(cm_savepath)

In [ ]:
def make_confusion_matrix_small(cm, class_names):
    fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
    plt.rcParams.update({'font.size': 6})
    ax = sns.heatmap(
        data=cm,
        annot=False,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names.values(),
        yticklabels=class_names.values(),
        ax=ax,
        vmax=100,
        vmin=0,
        cbar=False
      )
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

    return fig, ax

# Create confusion matrices for all combinations of models and test sets
for model in results:
  for test_set in results[model]:
    fig, ax = make_confusion_matrix_small(results[model][test_set]["confusion_matrix"], class_names)
    ax.set_title('')
    fig.savefig(os.path.join(cm_savepath, f'cm_{model}_{test_set}.png'), bbox_inches='tight')